# Hello, Semantic Axis Lab!

A minimal notebook to verify the local Jupyter → GitHub → Colab workflow. No additional packages or credentials are required.

Run the cell below with **Shift+Enter**. The saved output shows a previous run; rerunning it verifies your own environment.

In [1]:
import platform, sys

print("Hello, Semantic Axis Lab!")
print(f"Python {platform.python_version()}")
print("Notebook execution works.")
print(sys.executable)

Hello, Semantic Axis Lab!
Python 3.13.18
Notebook execution works.
/usr/bin/python3


In [9]:
%pip install transformers torch numpy plotly ipywidgets wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 55.1 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [wordfreq]


In [3]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "EleutherAI/pythia-70m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)
model.eval()

# Copy the frozen input embedding table to a NumPy array.
embeddings = (
    model.get_input_embeddings()
    .weight.detach()
    .cpu()
    .float()
    .numpy()
    .copy()
)

del model  # We only need the embedding table for now.

print("Embedding shape:", embeddings.shape)
print("Memory: %.1f MiB" % (embeddings.nbytes / 1024**2))

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  166MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

[transformers] GPTNeoXModel LOAD REPORT from: EleutherAI/pythia-70m
Key              | Status     |  | 
-----------------+------------+--+-
embed_out.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (50304, 512)
Memory: 98.2 MiB


In [5]:
for word in ["king", "queen", "man", "woman", "cat", "dog", "aunt", "uncle"]:
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    print(
        f"{word:8}",
        "IDs:", ids,
        "Tokens:", tokenizer.convert_ids_to_tokens(ids),
    )

king     IDs: [6963] Tokens: ['Ġking']
queen    IDs: [19538] Tokens: ['Ġqueen']
man      IDs: [637] Tokens: ['Ġman']
woman    IDs: [3416] Tokens: ['Ġwoman']
cat      IDs: [5798] Tokens: ['Ġcat']
dog      IDs: [4370] Tokens: ['Ġdog']
aunt     IDs: [25969] Tokens: ['Ġaunt']
uncle    IDs: [18796] Tokens: ['Ġuncle']


In [7]:
import numpy as np

# Normalize a separate copy for cosine-similarity searches.
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
unit_embeddings = embeddings / np.maximum(norms, 1e-12)

word = "cat"
ids = tokenizer.encode(" " + word, add_special_tokens=False)
assert len(ids) == 1, f"{word!r} uses multiple tokens: {ids}"

token_id = ids[0]
similarities = unit_embeddings @ unit_embeddings[token_id]
similarities[token_id] = -np.inf

nearest = np.argsort(similarities)[-30:][::-1]

for index in nearest:
    print(
        f"{int(index):6}  "
        f"{tokenizer.decode([int(index)])!r:24}  "
        f"{similarities[index]:.3f}"
    )

  8076  'cat'                     0.624
 12441  ' Cat'                    0.582
 23475  'Cat'                     0.579
 16581  ' cats'                   0.446
  4370  ' dog'                    0.362
 21048  ' Dog'                    0.279
  9097  ' dogs'                   0.277
 29286  ' pets'                   0.272
 17876  ' rabbit'                 0.270
  5893  ' animal'                 0.264
 12621  ' bird'                   0.263
  1429  ' child'                  0.253
 10464  ' bat'                    0.246
 12120  ' cow'                    0.245
 34664  ' poultry'                0.243
  2839  ' rat'                    0.242
 28005  ' monkey'                 0.240
 29466  ' chickens'               0.238
  7590  ' pet'                    0.238
 47074  'Dog'                     0.234
  6858  ' baby'                   0.234
 38091  ' puppy'                  0.233
 16396  ' sheep'                  0.227
  5006  ' boy'                    0.225
 24980  ' snake'                  0.225


In [8]:
query_id = tokenizer.encode(" cat", add_special_tokens=False)[0]
scores = unit_embeddings @ unit_embeddings[query_id]
scores[query_id] = -np.inf

order = np.argsort(scores)[::-1]
ranks = np.empty(len(order), dtype=int)
ranks[order] = np.arange(1, len(order) + 1)

for word in [
    "kitten", "kittens", "feline", "tiger", "lion",
    "leopard", "panther", "dog", "poultry",
]:
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    pieces = tokenizer.convert_ids_to_tokens(ids)

    if len(ids) == 1:
        i = ids[0]
        print(f"{word:10} rank={ranks[i]:5}  cosine={scores[i]:.3f}")
    else:
        print(f"{word:10} MULTI-TOKEN: {pieces}")

kitten     MULTI-TOKEN: ['Ġk', 'itten']
kittens    MULTI-TOKEN: ['Ġk', 'itt', 'ens']
feline     MULTI-TOKEN: ['Ġf', 'eline']
tiger      rank=   65  cosine=0.192
lion       rank=   42  cosine=0.211
leopard    MULTI-TOKEN: ['Ġle', 'opard']
panther    MULTI-TOKEN: ['Ġpant', 'her']
dog        rank=    5  cosine=0.362
poultry    rank=   15  cosine=0.243


In [11]:
import pandas as pd
from wordfreq import top_n_list

# Frequency-ranked English words; restrict this first audit to a–z.
words = [
    word
    for word in top_n_list("en", 30_000)
    if word.isascii() and word.isalpha() and word.islower()
][:10_000]

def token_info(text):
    ids = tokenizer.encode(text, add_special_tokens=False)
    return {
        "count": len(ids),
        "pieces": tokenizer.convert_ids_to_tokens(ids),
        "single": (
            len(ids) == 1
            and ids[0] not in tokenizer.all_special_ids
            and tokenizer.decode(ids) == text
        ),
    }

rows = []

for rank, word in enumerate(words, start=1):
    bare = token_info(word)
    spaced = token_info(" " + word)

    rows.append({
        "rank": rank,
        "word": word,
        "bare_single": bare["single"],
        "spaced_single": spaced["single"],
        "either_single": bare["single"] or spaced["single"],
        "both_single": bare["single"] and spaced["single"],
        "bare_pieces": bare["pieces"],
        "spaced_pieces": spaced["pieces"],
    })

coverage = pd.DataFrame(rows)

coverage["frequency_band"] = pd.cut(
    coverage["rank"],
    bins=[0, 1_000, 2_000, 5_000, 10_000],
    labels=["1–1,000", "1,001–2,000", "2,001–5,000", "5,001–10,000"],
)

columns = [
    "bare_single", "spaced_single",
    "either_single", "both_single",
]

summary = (
    coverage.groupby("frequency_band", observed=True)[columns]
    .mean()
    .mul(100)
    .round(1)
)

display(summary.rename(columns={
    "bare_single": "No leading space (%)",
    "spaced_single": "Leading space (%)",
    "either_single": "Either form (%)",
    "both_single": "Both forms (%)",
}))

print(f"Words tested: {len(coverage):,}")
print(
    "Overall coverage in either form:",
    f"{coverage['either_single'].mean():.1%}",
)

,No leading space (%),Leading space (%),Either form (%),Both forms (%)
frequency_band,,,,
"1–1,000",86.7,97.1,97.3,86.5
"1,001–2,000",55.9,94.4,94.8,55.5
"2,001–5,000",33.1,89.7,90.9,31.9
"5,001–10,000",17.2,82.5,84.0,15.6


Words tested: 10,000
Overall coverage in either form: 88.5%


In [12]:
from transformers import AutoTokenizer

LLAMA_ID = "meta-llama/Meta-Llama-3-8B"

llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_ID)

print("Tokenizer loaded.")
print("Vocabulary size:", len(llama_tokenizer))

for word in ["cat", "kitten", "kittens", "feline", "tiger", "leopard", "panther"]:
    for prefix in ["", " "]:
        text = prefix + word
        ids = llama_tokenizer.encode(text, add_special_tokens=False)
        print(f"{text!r:12} {len(ids)} token(s):",
              llama_tokenizer.convert_ids_to_tokens(ids))

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B.
401 Client Error. (Request ID: Root=1-6ae51c47-543b7875301daad17f4259f2;2a9e27b9-6df1-4d9a-9389-142a4340fbf3)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920/config.json.
Access to model meta-llama/Meta-Llama-3-8B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [13]:
import pandas as pd
from transformers import AutoTokenizer

tokenizers = {"Pythia-70M": tokenizer}

candidates = {
    "Qwen3-0.6B": "Qwen/Qwen3-0.6B",
    "SmolLM2-135M": "HuggingFaceTB/SmolLM2-135M",
}

for name, repo in candidates.items():
    print(f"Loading {name} tokenizer...")
    tokenizers[name] = AutoTokenizer.from_pretrained(repo)

Loading Qwen3-0.6B tokenizer...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (4 outputs left out)

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [14]:
def audit_tokenizer(tok, words):
    special_ids = set(tok.all_special_ids)

    def is_single(text):
        ids = tok.encode(text, add_special_tokens=False)
        return (
            len(ids) == 1
            and ids[0] not in special_ids
            and tok.decode(ids, clean_up_tokenization_spaces=False) == text
        )

    rows = []
    for rank, word in enumerate(words, start=1):
        bare = is_single(word)
        spaced = is_single(" " + word)
        rows.append({
            "rank": rank,
            "word": word,
            "bare": bare,
            "spaced": spaced,
            "either": bare or spaced,
            "both": bare and spaced,
        })

    result = pd.DataFrame(rows)
    result["band"] = pd.cut(
        result["rank"],
        bins=[0, 1000, 2000, 5000, 10000],
        labels=["1–1,000", "1,001–2,000",
                "2,001–5,000", "5,001–10,000"],
    )
    return result


audits = {
    name: audit_tokenizer(tok, words)
    for name, tok in tokenizers.items()
}

comparison = pd.DataFrame({
    name: audit.groupby("band", observed=True)["either"].mean() * 100
    for name, audit in audits.items()
})

print("Complete-word coverage in either form (%)")
display(comparison.round(1))

display(pd.DataFrame({
    name: {
        "Tokenizer size": len(tokenizers[name]),
        "Overall coverage (%)": round(audit["either"].mean() * 100, 1),
    }
    for name, audit in audits.items()
}).T)

Complete-word coverage in either form (%)


,Pythia-70M,Qwen3-0.6B,SmolLM2-135M
band,,,
"1–1,000",97.3,99.3,97.2
"1,001–2,000",94.8,97.9,94.6
"2,001–5,000",90.9,92.6,90.3
"5,001–10,000",84.0,87.3,83.1


,Tokenizer size,Overall coverage (%)
Pythia-70M,50277.0,88.5
Qwen3-0.6B,151669.0,91.1
SmolLM2-135M,49152.0,87.8


In [15]:
presence = pd.DataFrame({
    name: audit.set_index("word")["either"]
    for name, audit in audits.items()
})

qwen_only = presence.index[
    presence["Qwen3-0.6B"] & ~presence["Pythia-70M"]
]

print("Covered by Qwen but missing from Pythia:", len(qwen_only))
print(qwen_only[:80].tolist())

examples = [
    "cat", "kitten", "kittens", "feline",
    "tiger", "lion", "leopard", "panther",
]
display(presence.reindex(examples))

Covered by Qwen but missing from Pythia: 275
['american', 'york', 'london', 'america', 'april', 'british', 'september', 'david', 'james', 'august', 'china', 'november', 'french', 'european', 'michael', 'washington', 'december', 'australia', 'india', 'paul', 'chinese', 'russian', 'california', 'canada', 'friday', 'german', 'france', 'africa', 'sunday', 'christmas', 'indian', 'germany', 'japan', 'texas', 'christian', 'peter', 'robert', 'chicago', 'african', 'louis', 'paris', 'japanese', 'mike', 'monday', 'senate', 'joe', 'florida', 'mexico', 'martin', 'spanish', 'canadian', 'republican', 'muslim', 'italian', 'iran', 'jim', 'jewish', 'ohio', 'brazil', 'bbc', 'catholic', 'jr', 'pakistan', 'bible', 'nfl', 'haha', 'roman', 'dna', 'santa', 'latin', 'ltd', 'iphone', 'jane', 'morgan', 'vegas', 'congratulations', 'dean', 'arab', 'allen', 'berlin']


,Pythia-70M,Qwen3-0.6B,SmolLM2-135M
word,,,
cat,True,True,True
kitten,NaN,NaN,NaN
kittens,NaN,NaN,NaN
feline,NaN,NaN,NaN
tiger,True,True,True
lion,True,True,True
leopard,NaN,NaN,NaN
panther,NaN,NaN,NaN


In [16]:
examples = [
    "cat", "kitten", "kittens", "feline", "tiger",
    "lion", "leopard", "panther",
    "american", "london", "september", "christmas", "dna",
]

def single_token_forms(tok, word):
    matches = []
    special_ids = set(tok.all_special_ids)

    for spelling in dict.fromkeys([word, word.capitalize(), word.upper()]):
        for prefix in ["", " "]:
            text = prefix + spelling
            ids = tok.encode(text, add_special_tokens=False)

            if (
                len(ids) == 1
                and ids[0] not in special_ids
                and tok.decode(
                    ids, clean_up_tokenization_spaces=False
                ) == text
            ):
                matches.append(repr(text))

    return ", ".join(matches) or "None"

display(pd.DataFrame({
    name: {
        word: single_token_forms(tok, word)
        for word in examples
    }
    for name, tok in tokenizers.items()
}))

,Pythia-70M,Qwen3-0.6B,SmolLM2-135M
cat,"'cat', ' cat', 'Cat', ' Cat', 'CAT', ' CAT'","'cat', ' cat', 'Cat', ' Cat', 'CAT', ' CAT'","'cat', ' cat', 'Cat', ' Cat', 'CAT', ' CAT'"
kitten,None,' kitten',' kitten'
kittens,None,' kittens',' kittens'
feline,None,None,' feline'
tiger,"' tiger', ' Tiger'","' tiger', ' Tiger'","' tiger', ' Tiger'"
lion,"'lion', ' lion', ' Lion'","'lion', ' lion', ' Lion'","'lion', ' lion', ' Lion'"
leopard,None,"' leopard', ' Leopard'",' leopard'
panther,None,' Panther',None
american,"'American', ' American'","'american', ' american', 'American', ' American'","'american', ' american', 'American', ' American'"
london,"'London', ' London'","' london', 'London', ' London'","'London', ' London'"


In [17]:
case_coverage = pd.DataFrame({
    name: [
        single_token_forms(tok, word) != "None"
        for word in words
    ]
    for name, tok in tokenizers.items()
}, index=pd.Index(words, name="word"))

display(pd.DataFrame({
    "Lowercase coverage (%)": presence.mean() * 100,
    "Any tested capitalization (%)": case_coverage.mean() * 100,
}).round(1))

,Lowercase coverage (%),Any tested capitalization (%)
Pythia-70M,88.5,98.6
Qwen3-0.6B,91.1,99.8
SmolLM2-135M,87.8,96.8
